In [1]:
!pip install -q transformers datasets scikit-learn pandas torch huggingface_hub gspread gspread-dataframe google-auth

In [2]:
from google.colab import auth, userdata
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = userdata.get('SHEET_ID')
sheet = gc.open_by_key(SHEET_ID).get_worksheet(0)

import pandas as pd
data = sheet.get_all_records()
flywheel_df = pd.DataFrame(data)

print(f"Pulled {len(flywheel_df)} rows from Google Sheet")
print(flywheel_df.head())

Pulled 150 rows from Google Sheet
         Date                              Sender  \
0  03-05-2026  caroline.morgan@hollywoodwood.info   
1  03-03-2026          no-reply-forms@webflow.com   
2  02-20-2026          no-reply-forms@webflow.com   
3  02-11-2026          no-reply-forms@webflow.com   
4  02-03-2026               ari-kahn@capstra.info   

                                             Subject  \
0  Could be useful Gainesville Environmental Film...   
1  New General Contact Form submission at Cinema ...   
2     New Internship Form submission at Cinema Verde   
3  New Remote Volunteer form submission at Cinema...   
4  Quick thought Gainesville Environmental Film A...   

                                          Email Text  Label         Source  
0  Hey curious - Could a little financial support...   SPAM  Manual Import  
1  You just received a new General Contact Form s...   SPAM  Manual Import  
2  You just received a new Internship Form submis...  LEGIT  Manual Import  
3 

In [3]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

#use sheet as single source of truth
combined_df = flywheel_df.copy()

combined_df['Combined_Text'] = "Sender: " + combined_df['Sender'].astype(str) + " | Subject: " + combined_df['Subject'].astype(str) + " | Body: " + combined_df['Email Text'].astype(str)
combined_df = combined_df[['Combined_Text', 'Label']]
combined_df['Label'] = combined_df['Label'].map({'LEGIT': 0, 'SPAM': 1})

combined_df = combined_df.dropna(subset=['Label'])
combined_df['Label'] = combined_df['Label'].astype(int)

print(f"Total dataset: {len(combined_df)} rows")
print(f"Legit: {len(combined_df[combined_df['Label'] == 0])}, Spam: {len(combined_df[combined_df['Label'] == 1])}")

Total dataset: 150 rows
Legit: 100, Spam: 50


In [4]:
from transformers import AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

X_train, X_test, y_train, y_test = train_test_split(
    combined_df['Combined_Text'],
    combined_df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=combined_df['Label']
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

train_df = pd.DataFrame({'text': X_train.values, 'label': y_train.values})
test_df = pd.DataFrame({'text': X_test.values, 'label': y_test.values})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_train = train_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print("Tokenization complete.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 120, Test: 30


Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results_retrain",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Training...")
trainer.train()

print("\nEvaluating...")
results = trainer.evaluate()
print(f"Retrain Metrics: {results}")

# confusion matrix
predictions = trainer.predict(tokenized_test)
y_pred = predictions.predictions.argmax(-1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit (0)', 'Spam (1)'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Retrained Model: Confusion Matrix")
plt.show()

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

trainer.save_model("/content/retrained-model")
tokenizer.save_pretrained("/content/retrained-model")

model.push_to_hub("evanastevska/cinema-verde-spam-classifier")
tokenizer.push_to_hub("evanastevska/cinema-verde-spam-classifier")

print("Retrained model pushed to Hugging Face.")